In [1]:
!pip -q install -U transformers accelerate sentencepiece safetensors
!pip -q install -U pymupdf pdfminer.six

!pip -q install -U datasets evaluate rouge-score bert-score ipywidgets trl

import torch, platform
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 51.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.9 MB/s eta 0:00:00
Torch: 2.8.0+cu126 | CUDA: True


## Secrets


In [3]:
import os

def _mask(token: str, left: int = 6, right: int = 4) -> str:
    token = token or ""
    if len(token) <= left + right:
        return "*" * len(token)
    return token[:left] + "…" + token[-right:]

def _from_colab(name: str) -> str:
    """Fetch a secret from Colab's Secrets (UI: left sidebar → Secrets)."""
    try:
        from google.colab import userdata  # type: ignore
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""

OPENAI_API_KEY = _from_colab("openaikey")
HF_TOKEN       = _from_colab("huggingfacekey")

if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
if HF_TOKEN:
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN

print("OpenAI key set:", bool(OPENAI_API_KEY), _mask(OPENAI_API_KEY) if OPENAI_API_KEY else "")
print("HF token set  :", bool(HF_TOKEN),       _mask(HF_TOKEN)       if HF_TOKEN else "")

if HF_TOKEN:
    try:
        from huggingface_hub import login, whoami
        login(token=HF_TOKEN)
        user = whoami()
        print("HF user:", user.get("name") or user.get("email") or user.get("id"))
    except Exception as e:
        print("Hugging Face login failed:", e)


OpenAI key set: True sk-pro…_mcA
HF token set  : True hf_bDa…EhQq
HF user: CourierCat


## Config



In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class CFG:
    data_dir: str = "/content/drive/MyDrive/MLE_In_Gen_AI/data/arxiv"
    max_papers: int = 50
    max_chars_per_doc: int = 12000

    gen_model_name: str = "Qwen/Qwen2.5-3B-Instruct"
    gen_max_new_tokens: int = 200
    gen_temp_1: float = 0.2
    gen_temp_2: float = 0.8

    rm_base: str = "microsoft/deberta-v3-base"
    rm_output_dir: str = "rm_ckpt"
    rm_epochs: int = 1
    rm_lr: float = 3e-5
    rm_bsz: int = 8
    rm_grad_accum: int = 2

cfg = CFG()
cfg

CFG(data_dir='/content/drive/MyDrive/MLE_In_Gen_AI/data/arxiv', max_papers=50, max_chars_per_doc=12000, gen_model_name='Qwen/Qwen2.5-3B-Instruct', gen_max_new_tokens=200, gen_temp_1=0.2, gen_temp_2=0.8, rm_base='microsoft/deberta-v3-base', rm_output_dir='rm_ckpt', rm_epochs=1, rm_lr=3e-05, rm_bsz=8, rm_grad_accum=2)

## Mounting Google Drive

In [5]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
Path(cfg.data_dir).mkdir(parents=True, exist_ok=True)

pdfs = sorted(Path(cfg.data_dir).glob("*.pdf"))
print(f"Drive mounted. Reading from: {cfg.data_dir}")
print(f"Found {len(pdfs)} PDFs (showing up to 5):")
for fp in pdfs[:5]:
    print(" -", fp.name)


Mounted at /content/drive
Drive mounted. Reading from: /content/drive/MyDrive/MLE_In_Gen_AI/data/arxiv
Found 50 PDFs (showing up to 5):
 - 2508.10507v1 - Multi-Sample Anti-Aliasing and Constrained Optimization for 3D Gaussian Splatting.pdf
 - 2508.10528v1 - Med-GLIP Advancing Medical Language-Image Pre-training with Large-scale Grounded Dataset.pdf
 - 2508.10530v1 - Diversity First Quality Later A Two-Stage Assumption for Language Model Alignment.pdf
 - 2508.10539v1 - Improving Value-based Process Verifier via Low-Cost Variance Reduction.pdf
 - 2508.10548v1 - Stabilizing Long-term Multi-turn Reinforcement Learning with Gated Rewards.pdf


## 3. Load & extract from dataset


In [6]:
import re
from pathlib import Path

def _clean_text(t: str) -> str:
    t = t.replace("\x00", " ")
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

# Extract with PyMuPDF, fall back to pdfminer.six
def extract_text_pdf(path: Path) -> str:
    # 1) PyMuPDF
    try:
        import fitz  # pymupdf
        doc = fitz.open(path)
        parts = [page.get_text() for page in doc]
        doc.close()
        text = "\n".join(parts)
        if text.strip():
            return _clean_text(text)
    except Exception as e:
        print(f"PyMuPDF failed on {path.name}: {e}")

    # 2) pdfminer.six
    try:
        from pdfminer.high_level import extract_text as pdfminer_extract
        text = pdfminer_extract(str(path)) or ""
        if text.strip():
            return _clean_text(text)
    except Exception as e:
        print(f"pdfminer failed on {path.name}: {e}")

    return ""

pdf_paths = sorted(Path(cfg.data_dir).glob("*.pdf"))[: cfg.max_papers]
if not pdf_paths:
    raise FileNotFoundError(f"No PDFs found in {cfg.data_dir}")

dataset = []
for p in pdf_paths:
    txt = extract_text_pdf(p)
    if not txt:
        print(f"Could not extract {p.name}")
        continue
    txt = txt[: cfg.max_chars_per_doc]
    dataset.append({"context": f"Title: {p.stem}\n\n{txt}", "reference": ""})

if not dataset:
    raise RuntimeError("No usable documents after extraction. Check your PDFs or install extraction libs.")

print(f"Prepared {len(dataset)} documents.")


Prepared 50 documents.


## Loading model

In [7]:
import os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

HF_TOKEN = os.getenv("HUGGINGFACE_HUB_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")

PREFERRED = cfg.gen_model_name
FALLBACKS = [
    "Qwen/Qwen2.5-1.5B-Instruct",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "gpt2",
]

def _load(repo_id: str):
    kw = dict(trust_remote_code=True)
    if HF_TOKEN: kw["token"] = HF_TOKEN
    tok = AutoTokenizer.from_pretrained(repo_id, use_fast=True, **kw)
    model = AutoModelForCausalLM.from_pretrained(
        repo_id,
        torch_dtype=(torch.float16 if torch.cuda.is_available() else torch.float32),
        device_map="auto",
        **kw,
    )
    return tok, model

repo = PREFERRED
try:
    tok, model = _load(repo)
    print(f"Loaded {repo}")
except Exception as e:
    print(f"Failed {repo}: {e}\n→ Trying fallbacks…")
    last = e
    tok = model = None
    for r in FALLBACKS:
        try:
            tok, model = _load(r); repo = r
            print(f"Loaded fallback {repo}")
            break
        except Exception as e2:
            last = e2
    if model is None:
        raise RuntimeError(f"All model loads failed. Last error:\n{last}")

cfg.gen_model_name = repo


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-3B-Instruct


## Generate 2 summaries per doc


In [8]:
import random, transformers

SYS = "You are a helpful research assistant. Write a concise, faithful summary. Avoid adding facts not supported by the text."
USR_TMPL = """Summarize the following content for a technically literate audience in 3–5 sentences:

{context}
"""

def _build_prompt(messages):
    if hasattr(tok, "apply_chat_template"):
        try:
            return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except TypeError:
            pass

    sys = messages[0]["content"]
    usr = messages[1]["content"]
    return f"<s>[INST] <<SYS>>\n{sys}\n<</SYS>>\n\n{usr} [/INST]"

def gen_summary(context, temp=0.2, seed=0):
    random.seed(seed); torch.manual_seed(seed)
    # Trim context cautiously
    context = context[-4096:]
    messages = [
        {"role": "system", "content": SYS},
        {"role": "user",   "content": USR_TMPL.format(context=context)},
    ]
    prompt = _build_prompt(messages)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    eos = tok.eos_token_id if tok.eos_token_id is not None else getattr(model.generation_config, "eos_token_id", None)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=cfg.gen_max_new_tokens,
            do_sample=True,
            temperature=temp,
            top_p=0.95,
            eos_token_id=eos,
        )
    return tok.decode(out[0], skip_special_tokens=True).strip()

pairs = []
for i, ex in enumerate(dataset):
    c1 = gen_summary(ex["context"], temp=cfg.gen_temp_1, seed=42+i)
    c2 = gen_summary(ex["context"], temp=cfg.gen_temp_2, seed=99+i)
    pairs.append({"context": ex["context"], "reference": ex["reference"], "cand_a": c1, "cand_b": c2})
    print(f"Item {i}: generated two candidates.")
len(pairs)


Item 0: generated two candidates.
Item 1: generated two candidates.
Item 2: generated two candidates.
Item 3: generated two candidates.
Item 4: generated two candidates.
Item 5: generated two candidates.
Item 6: generated two candidates.
Item 7: generated two candidates.
Item 8: generated two candidates.
Item 9: generated two candidates.
Item 10: generated two candidates.
Item 11: generated two candidates.
Item 12: generated two candidates.
Item 13: generated two candidates.
Item 14: generated two candidates.
Item 15: generated two candidates.
Item 16: generated two candidates.
Item 17: generated two candidates.
Item 18: generated two candidates.
Item 19: generated two candidates.
Item 20: generated two candidates.
Item 21: generated two candidates.
Item 22: generated two candidates.
Item 23: generated two candidates.
Item 24: generated two candidates.
Item 25: generated two candidates.
Item 26: generated two candidates.
Item 27: generated two candidates.
Item 28: generated two candida

50

## 5. Choose the better summary


In [44]:
!pip -q install -U ipywidgets==8.1.2
from google.colab import output
output.enable_custom_widget_manager()

import ipywidgets as w
from IPython.display import display
display(w.Button(description="Widget test"))


Button(description='Widget test', style=ButtonStyle())

In [50]:
import os, json, pathlib, time, re
from collections import Counter

assert 'pairs' in globals() and len(pairs) > 0, "Generate `pairs` first."
pathlib.Path("labels").mkdir(exist_ok=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
USE_ASSISTANT  = bool(OPENAI_API_KEY)

def _kw(t):
    t = re.sub(r"[^a-z0-9\s]+"," ", t.lower())
    stop = {"the","and","for","that","with","this","from","into","your","have","been","were","their","there",
            "these","those","using","about","more","most","very","also","over","under","between","within"}
    return [x for x in t.split() if len(x) > 2 and x not in stop]

def _heuristic_pick(ex):
    # Title keyword overlap; tie-breaker = shorter summary
    title = ex["context"].splitlines()[0].replace("Title:","").strip()
    key = Counter(_kw(title))
    def score(s):
        toks = _kw(s)
        return (sum(key[t] for t in toks), -len(toks))
    a, b = score(ex["cand_a"]), score(ex["cand_b"])
    label = "A" if a > b else "B"
    reason = f"Heuristic: {'more' if (a[0]!=b[0]) else 'equal'} title overlap; {'A' if a>b else 'B'} preferred, tie-break by length if equal."
    return label, reason

def _assistant_pick(ex):
    # Uses OpenAI Chat Completions to choose A or B with a brief reason
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    sys_msg = (
        "Judge which summary (A or B) better matches the given context. "
        "Prefer factual correctness, coverage of key points, and clarity. "
        "Respond ONLY as strict JSON: {\"label\":\"A|B\",\"reason\":\"...\"}."
    )
    usr = (
        "Context:\n" + ex["context"][:6000] +  # safety cap
        "\n\nSummary A:\n" + ex["cand_a"] +
        "\n\nSummary B:\n" + ex["cand_b"] +
        "\n\nReturn JSON ONLY."
    )
    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role":"system","content":sys_msg},
                      {"role":"user","content":usr}],
            temperature=0.0,
            response_format={"type":"json_object"},
        )
        raw = resp.choices[0].message.content or "{}"
        obj = json.loads(raw)
        label = obj.get("label", "").strip().upper()
        reason = obj.get("reason", "").strip()
        if label not in ("A","B"):
            raise ValueError("Bad label")
        return label, reason
    except Exception as e:
        # On any issue, fall back to heuristic
        hlabel, hreason = _heuristic_pick(ex)
        return hlabel, f"Fallback to heuristic due to API error: {e}. {hreason}"

saved = 0
for i, ex in enumerate(pairs):
    if os.path.exists(f"labels/pair_{i:03d}.json"):
        continue
    if USE_ASSISTANT:
        label, reason = _assistant_pick(ex)
        time.sleep(0.3)
    else:
        label, reason = _heuristic_pick(ex)

    rec = {
        "context":   ex["context"],
        "chosen":    ex["cand_a"] if label == "A" else ex["cand_b"],
        "rejected":  ex["cand_b"] if label == "A" else ex["cand_a"],
        "reference": ex.get("reference", ""),
        "pair_id":   i,
        "assistant_label": label,
        "assistant_reason": reason,
    }
    with open(f"labels/pair_{i:03d}.json","w",encoding="utf-8") as f:
        json.dump(rec, f, ensure_ascii=False)
    saved += 1
    if (i+1) % 5 == 0:
        print(f"Labeled up to {i+1}/{len(pairs)}")

print(f"Done. Wrote {saved} new label files to ./labels (assistant_used={USE_ASSISTANT}).")


Labeled up to 5/50
Labeled up to 10/50
Labeled up to 15/50
Labeled up to 20/50
Labeled up to 25/50
Labeled up to 30/50
Labeled up to 35/50
Labeled up to 40/50
Labeled up to 45/50
Labeled up to 50/50
Done. Wrote 49 new label files to ./labels (assistant_used=True).


## Label and save preferences


In [51]:
import ipywidgets as w, json, pathlib
from IPython.display import display, Markdown

pathlib.Path("labels").mkdir(exist_ok=True)

idx = w.IntText(value=0, description="Index", min=0, max=len(pairs)-1)
lbl = w.ToggleButtons(options=[("A better","A"),("B better","B")], description="Preference")
save = w.Button(description="Save label")
out = w.Output()
status = w.Label()

def render(i):
    out.clear_output()
    with out:
        ex = pairs[i]
        display(Markdown(f"### Example {i}"))
        display(Markdown("**Context (truncated)**"))
        display(Markdown(f"<pre style='white-space:pre-wrap'>{ex['context'][:3000]}</pre>"))
        display(Markdown("**Candidate A**")); display(Markdown(ex["cand_a"]))
        display(Markdown("**Candidate B**")); display(Markdown(ex["cand_b"]))

def on_idx_change(change):
    render(change["new"])

def on_save(_):
    i = idx.value
    pref = lbl.value
    if not pref:
        status.value = "Select a preference first."
        return
    ex = pairs[i]
    rec = {
        "context": ex["context"],
        "chosen": ex["cand_a"] if pref=="A" else ex["cand_b"],
        "rejected": ex["cand_b"] if pref=="A" else ex["cand_a"],
        "reference": ex.get("reference",""),
        "pair_id": i,
    }
    fn = f"labels/pair_{i:03d}.json"
    with open(fn, "w", encoding="utf-8") as f:
        json.dump(rec, f, ensure_ascii=False)
    status.value = f"Saved {fn}"

idx.observe(on_idx_change, names="value")
save.on_click(on_save)
display(w.VBox([idx, lbl, save, status, out]))
render(0)


## Build reward data


In [52]:
import glob, json
files = sorted(glob.glob("labels/pair_*.json"))
assert files, "No labels found. Use the widget to save some first."
with open("reward_data.jsonl", "w", encoding="utf-8") as f:
    for fp in files:
        rec = json.load(open(fp, "r", encoding="utf-8"))
        f.write(json.dumps({"chosen": rec["chosen"], "rejected": rec["rejected"]}, ensure_ascii=False) + "\n")
print("Wrote reward_data.jsonl with", len(files), "pairs")


Wrote reward_data.jsonl with 50 pairs


## Train a small reward model

In [56]:
!pip -q install "trl==0.8.6"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 kB 5.5 MB/s eta 0:00:00


In [64]:
import gc, torch
# If you created a generator earlier (e.g., `model` / `tok`)
for obj in ["model", "tok", "pipe", "generator", "text_model"]:
    if obj in globals():
        try:
            del globals()[obj]
        except:
            pass
gc.collect()
try:
    torch.cuda.empty_cache()
except:
    pass


In [ ]:
cfg.rm_base = "microsoft/deberta-v3-small"

MAX_LEN = 256
BATCH_SIZE = 1
GRAD_ACCUM = 4

from transformers import AutoTokenizer
tok_rm = AutoTokenizer.from_pretrained(cfg.rm_base, use_fast=True)

def tok_pairwise(batch):
    ch = tok_rm(batch["chosen"],   truncation=True, padding="max_length", max_length=MAX_LEN)
    rj = tok_rm(batch["rejected"], truncation=True, padding="max_length", max_length=MAX_LEN)
    return {
        "input_ids_chosen": ch["input_ids"],
        "attention_mask_chosen": ch["attention_mask"],
        "input_ids_rejected": rj["input_ids"],
        "attention_mask_rejected": rj["attention_mask"],
    }

tokenized = dataset_rm.map(tok_pairwise, batched=True, remove_columns=["chosen","rejected"])

import numpy as np, torch
def collate_fn(features):
    def stack(key): return torch.tensor(np.array([f[key] for f in features]), dtype=torch.long)
    return {
        "input_ids_chosen":        stack("input_ids_chosen"),
        "attention_mask_chosen":   stack("attention_mask_chosen"),
        "input_ids_rejected":      stack("input_ids_rejected"),
        "attention_mask_rejected": stack("attention_mask_rejected"),
    }


## Evaluation

In [68]:
import re, glob, json, os, textwrap

def extract_abstract(context: str) -> str:
    txt = Context
    first_nl = txt.find("\n")
    if first_nl != -1:
        txt = txt[first_nl+1:]

    # Look for "Abstract" heading
    m = re.search(r"\babstract\b[:\s]*", txt, flags=re.IGNORECASE)
    if m:
        start = m.end()
        # stop at common next section headers
        stop_pat = re.compile(
            r"\n\s*(?:\d+\s+)?(?:introduction|background|related\s+work|1\.\s*introduction)\b",
            flags=re.IGNORECASE
        )
        stop_m = stop_pat.search(txt, start)
        end = stop_m.start() if stop_m else start + 1800
        abstract = txt[start:end].strip()
        abstract = re.sub(r"[ \t]+", " ", abstract)
        abstract = re.sub(r"\n{2,}", "\n", abstract)
        abstract = abstract.strip()
        if len(abstract) > 1000:
            abstract = abstract[:1000]
        if len(abstract.split()) >= 20:
            return abstract

    # Fallback: first 4–5 sentences from the body
    body = txt.strip()
    sents = re.split(r"(?<=[.!?])\s+", body)
    ref = " ".join(sents[:5]).strip()
    return textwrap.shorten(ref, width=1000, placeholder=" …")

updated = 0
files = sorted(glob.glob("labels/pair_*.json"))
for fp in files:
    with open(fp, "r", encoding="utf-8") as f:
        rec = json.load(f)
    if not rec.get("reference"):
        rec["reference"] = extract_abstract(rec["context"])
        with open(fp, "w", encoding="utf-8") as f:
            json.dump(rec, f, ensure_ascii=False)
        updated += 1

print(f"Backfilled references in {updated} of {len(files)} label files.")


Backfilled references in 50 of 50 label files.


In [69]:
from evaluate import load
import glob, json as _json, numpy as np, os, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Collect predictions and references
pairs_files = sorted(glob.glob("labels/pair_*.json"))
preds_ch, preds_rj, refs = [], [], []
for fp in pairs_files:
    rec = _json.load(open(fp,"r",encoding="utf-8"))
    preds_ch.append(rec["chosen"])
    preds_rj.append(rec["rejected"])
    refs.append(rec.get("reference",""))

valid_idx = [i for i,r in enumerate(refs) if isinstance(r,str) and r.strip()]

if valid_idx:
    rouge = load("rouge")
    bertscore = load("bertscore")
    rc = rouge.compute(
        predictions=[preds_ch[i] for i in valid_idx],
        references=[refs[i] for i in valid_idx],
    )
    rr = rouge.compute(
        predictions=[preds_rj[i] for i in valid_idx],
        references=[refs[i] for i in valid_idx],
    )
    print("ROUGE (chosen):  ", rc)
    print("ROUGE (rejected):", rr)

    bc = bertscore.compute(
        predictions=[preds_ch[i] for i in valid_idx],
        references=[refs[i] for i in valid_idx],
        lang="en",
    )
    br = bertscore.compute(
        predictions=[preds_rj[i] for i in valid_idx],
        references=[refs[i] for i in valid_idx],
        lang="en",
    )
    print("BERTScore F1 mean (chosen):  ", float(np.mean(bc["f1"])))
    print("BERTScore F1 mean (rejected):", float(np.mean(br["f1"])))
else:
    print("Still no non-empty references after backfill.")

# ---------- Reward-model agreement (loads a valid checkpoint or falls back) ----------
def _has_valid_checkpoint(path: str) -> bool:
    return os.path.exists(os.path.join(path, "config.json"))

def _find_ckpt_dir() -> str:
    base = getattr(cfg, "rm_output_dir", "rm_ckpt")
    if os.path.isdir(base):
        sub = sorted(glob.glob(os.path.join(base, "checkpoint-*")), key=os.path.getmtime, reverse=True)
        for c in sub:
            if _has_valid_checkpoint(c):
                return c
        if _has_valid_checkpoint(base):
            return base
    return cfg.rm_base

rm_dir = _find_ckpt_dir()
print("Scoring with reward model from:", rm_dir)

tok_rm = AutoTokenizer.from_pretrained(rm_dir, use_fast=True)
model_rm = AutoModelForSequenceClassification.from_pretrained(rm_dir)
device = "cuda" if torch.cuda.is_available() else "cpu"
model_rm.to(device).eval()

def score(texts, bs=8, max_len=256):
    out = []
    for i in range(0, len(texts), bs):
        batch = tok_rm(texts[i:i+bs], return_tensors="pt", padding=True, truncation=True, max_length=max_len).to(device)
        with torch.no_grad():
            out.append(model_rm(**batch).logits.squeeze(-1).detach().cpu().numpy())
    return np.concatenate(out) if out else np.array([])

s_ch = score(preds_ch); s_rj = score(preds_rj)
if len(s_ch) and len(s_rj):
    acc = float((s_ch > s_rj).mean())
    print(f"Reward model preference accuracy (chosen > rejected): {acc:.3f}")


ROUGE (chosen):   {'rouge1': np.float64(0.18743361154452334), 'rouge2': np.float64(0.050710788575234464), 'rougeL': np.float64(0.0908753467192501), 'rougeLsum': np.float64(0.1824087056231199)}
ROUGE (rejected): {'rouge1': np.float64(0.18739778615182712), 'rouge2': np.float64(0.050237287345084855), 'rougeL': np.float64(0.08957868785265927), 'rougeLsum': np.float64(0.18180992931540077)}


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore F1 mean (chosen):   0.8132817924022675
BERTScore F1 mean (rejected): 0.8132817924022675
Scoring with reward model from: microsoft/deberta-v3-small


/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Reward model preference accuracy (chosen > rejected): 0.000


## Artifacts


In [70]:

import os, glob, json, tarfile, pathlib, shutil
print("Files in CWD:")
print("\n".join(sorted(os.listdir("."))))


Files in CWD:
.config
drive
labels
reward_data.jsonl
rm_ckpt
sample_data
wandb


In [72]:
from google.colab import files
files.download("project_export.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>